In [0]:
%sql
SELECT email, user_country, regulation, created_at
FROM shopping_assistant.raw_logs.customer_pii
WHERE created_at >= current_timestamp() - INTERVAL 30 MINUTES
ORDER BY created_at DESC;

In [0]:
%sql
DELETE FROM shopping_assistant.raw_logs.customer_pii
WHERE email = 'noopurk414@gmail.com';

In [0]:
def _parse_country_from_locale(locale):
    if not locale:
        return ""
    parts = locale.replace("_", "-").split("-")
    if len(parts) >= 2 and len(parts[-1]) == 2:
        return parts[-1].upper()
    return ""

tests = ["en-GB", "en-US", "en-IN", "pt-BR", "es-MX", "de-DE", "en", "", None]
for loc in tests:
    print(f"{loc!r:10} -> {_parse_country_from_locale(loc or '')!r}")

In [0]:
%sql
SELECT
    statement_text,
    execution_status,
    error_message,
    start_time
FROM system.query.history
WHERE start_time >= current_timestamp() - INTERVAL 15 MINUTES
  AND statement_text LIKE '%ai_interactions_raw%'
ORDER BY start_time DESC;

In [0]:
def _fire(table: str, row: dict, on_failure=None) -> None:
    """Fire-and-forget background write — never blocks the caller."""
    threading.Thread(target=_write_row, args=(table, row, on_failure), daemon=True).start()

In [0]:
result = wrapper.log_interaction(
    user_email   = "test.gap05@example.com",
    user_input   = "test query for G-05 verification",
    model_output = "test response",
    model_name   = "test-model",
    status       = "success",
)
print(result)

In [0]:
import os
os.environ["DATABRICKS_SQL_WAREHOUSE_ID"] = "8eba55b5eb828697"

from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(
    app_id  = "myre_app",
    catalog = "nonexistent_catalog_xyz",
)

In [0]:
from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(
    app_id  = "myre_app",
    catalog = "nonexistent_catalog_xyz",   # deliberately wrong, so the registry check fails
)

In [0]:
%sql
SELECT
    i.session_id,
    i.trace_id,
    i.input_text_sanitised   AS user_input,
    i.status,
    i.request_timestamp,
    MAX(CASE WHEN g.policy_name = 'input_content_safety'  THEN g.result END)          AS input_guardrail_result,
    MAX(CASE WHEN g.policy_name = 'input_content_safety'  THEN g.triggered_block END) AS input_blocked,
    MAX(CASE WHEN g.policy_name = 'output_content_safety' THEN g.result END)          AS output_guardrail_result,
    MAX(CASE WHEN g.policy_name = 'output_content_safety' THEN g.triggered_block END) AS output_blocked
FROM shopping_assistant.raw_logs.ai_interactions_raw i
LEFT JOIN shopping_assistant.raw_logs.guardrail_results_raw g
    ON i.trace_id = g.trace_id
WHERE i.request_timestamp >= current_timestamp() - INTERVAL 15 MINUTES
GROUP BY i.session_id, i.trace_id, i.input_text_sanitised, i.status, i.request_timestamp;

In [0]:
%sql
SELECT session_id, trace_id, input_text_sanitised, status, request_timestamp
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE request_timestamp >= current_timestamp() - INTERVAL 15 MINUTES
ORDER BY request_timestamp DESC;

In [0]:
%sql
SELECT
    i.session_id,
    i.trace_id,
    i.input_text_sanitised   AS user_input,
    i.status,
    i.request_timestamp,
    MAX(CASE WHEN g.policy_name = 'input_content_safety'  THEN g.result END)          AS input_guardrail_result,
    MAX(CASE WHEN g.policy_name = 'input_content_safety'  THEN g.triggered_block END) AS input_blocked,
    MAX(CASE WHEN g.policy_name = 'output_content_safety' THEN g.result END)          AS output_guardrail_result,
    MAX(CASE WHEN g.policy_name = 'output_content_safety' THEN g.triggered_block END) AS output_blocked
FROM shopping_assistant.raw_logs.ai_interactions_raw i
LEFT JOIN shopping_assistant.raw_logs.guardrail_results_raw g
    ON i.trace_id = g.trace_id
WHERE i.session_id = '4bc8fcff-42c7-433e-ab6e-448d32375d42'
GROUP BY i.session_id, i.trace_id, i.input_text_sanitised, i.status, i.request_timestamp

In [0]:
%sql
SELECT trace_id, policy_name, result, triggered_block, checked_at
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE checked_at >= current_timestamp() - INTERVAL 15 MINUTES
ORDER BY checked_at DESC;

In [0]:
%sql
SELECT
    g.trace_id,
    g.policy_name,
    g.result,
    g.triggered_block,
    g.checked_at,
    i.session_id,
    i.input_text_sanitised,
    i.status
FROM shopping_assistant.raw_logs.guardrail_results_raw g
LEFT JOIN shopping_assistant.raw_logs.ai_interactions_raw i
    ON g.trace_id = i.trace_id
WHERE g.checked_at >= current_timestamp() - INTERVAL 30 MINUTES
ORDER BY g.checked_at DESC;

In [0]:
%sql
SELECT trace_id, policy_name, result, score, triggered_block, checked_at
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE checked_at >= current_timestamp() - INTERVAL 30 MINUTES
ORDER BY checked_at DESC;

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT trace_id, tool_name, status
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '9d6e778c-0648-4f99-b103-43186e38b070'

In [0]:
%sql
SELECT trace_id, tool_name, status
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '9d6e778c-0648-4f99-b103-43186e38b070'

In [0]:
%sql
SELECT trace_id, input_text_sanitised, session_id, request_timestamp
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY request_timestamp DESC
LIMIT 10

In [0]:
%sql
SELECT trace_id, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
!grep -rn "audit\.\|'audit'\|\"audit\"\|guardrail_results\b\|tool_calls\b" \
  --include="*.py" . | grep -v "_raw"

In [0]:
%sql
SELECT timestamp_seconds(floor(unix_timestamp(start_time) / 10) * 10) AS bucket,
       COUNT(*) AS statements,
       SUM(CASE WHEN statement_type = 'INSERT' THEN 1 ELSE 0 END) AS inserts,
       SUM(CASE WHEN execution_status = 'FAILED' THEN 1 ELSE 0 END) AS failed,
       MAX(waiting_at_capacity_duration_ms) AS max_queue_ms
FROM system.query.history
WHERE compute.warehouse_id = '8eba55b5eb828697'
  AND executed_by = '28c6a177-7992-4baa-9c88-b6cacc1152b3'
  AND start_time >= '2026-08-26'
  AND (error_message IS NULL OR error_message NOT LIKE '%nonexistent_catalog_xyz%')
GROUP BY 1
HAVING COUNT(*) > 3
ORDER BY 1

In [0]:
%sql
SELECT statement_type,
       execution_status,
       regexp_extract(error_message, '`?(\\w+)`?\\.`?(\\w+)`?\\.`?(\\w+)`? cannot be found', 0) AS missing_table,
       CASE WHEN error_message LIKE '%UNRESOLVED_COLUMN%' THEN 'version column' END AS col_error,
       COUNT(*) AS n
FROM system.query.history
WHERE compute.warehouse_id = '8eba55b5eb828697'
  AND executed_by = '28c6a177-7992-4baa-9c88-b6cacc1152b3'
  AND start_time >= '2026-08-26'
GROUP BY ALL
ORDER BY n DESC

In [0]:
%sql
SELECT statement_id, statement_type, executed_by, execution_status,
       start_time, total_duration_ms,
       waiting_for_compute_duration_ms, waiting_at_capacity_duration_ms,
       compute.warehouse_id, error_message
FROM system.query.history
WHERE compute.warehouse_id = '8eba55b5eb828697'
  AND start_time >= current_timestamp() - INTERVAL 3 HOURS
ORDER BY start_time DESC

In [0]:
test_statement_id = None
result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = """
        INSERT INTO shopping_assistant.raw_logs.logging_failures
        (failure_id, app_id, trace_id, failed_table, error_message, occurred_at, recovered, created_at, schema_version)
        VALUES ('test-insert-check-001', 'myre_app', 'test-trace-insert-check', 'test_table', 'TEST INSERT for AT-PERF-04 diagnosis', current_timestamp(), false, current_timestamp(), '1.0')
    """,
    wait_timeout = "10s",
)
print("Statement ID:", result.statement_id)
print("Status:", result.status.state)

In [0]:
w.secrets.get_secret(scope="shopping_assistant", key="databricks_sql_warehouse_id")

In [0]:
import base64
secret = w.secrets.get_secret(scope="shopping_assistant", key="databricks_sql_warehouse_id")
try:
    print(base64.b64decode(secret.value).decode("utf-8"))
except:
    print(secret.value)

In [0]:
result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = """
        SELECT 1 as test_marker
    """,
    wait_timeout = "10s",
)
print("Statement ID:", result.statement_id)
print("Status:", result.status.state)

In [0]:
%sql
SELECT trace_id, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id IN (
    'e3e40dc3-4a3e-4ba4-a427-775e436f0116',
    '81db2363-418c-4fe4-8d43-d0376f4cb914',
    '101ccdfe-07ae-4e37-b06d-f3327ea06dea',
    'e9e370bd-e35c-4b73-ad9e-1eb6d29f99a5'
)
ORDER BY created_at

In [0]:
%sql
SELECT trace_id, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, request_timestamp, created_at,
  (unix_timestamp(created_at) - unix_timestamp(request_timestamp)) as lag_seconds
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC
LIMIT 20

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")
print("Restored — AUDIT_APP_ID set back to myre_app")

In [0]:
w.secrets.delete_secret(scope="shopping_assistant", key="audit-app-id")
print("Deleted — audit now disabled")

In [0]:
%sql
SELECT trace_id, latency_ms, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE input_text_sanitised = 'show me black bags'
ORDER BY created_at DESC LIMIT 10

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.customer_pii

In [0]:
%sql
SELECT trace_id, input_text_sanitised, status
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 10

In [0]:
%sql
SELECT trace_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw

In [0]:
%sql
SELECT trace_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 5

In [0]:
%sql
SELECT trace_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 3

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-catalog", string_value="shopping_assistant")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-catalog", string_value="nonexistent_catalog_xyz")

In [0]:
%sql
SELECT node_name, started_at, ended_at, latency_ms
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'
ORDER BY started_at

In [0]:
%sql
SELECT COUNT(*) as customer_pii_count
FROM shopping_assistant.raw_logs.customer_pii
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT 'sessions' as tbl, COUNT(DISTINCT s.subject_ref) as distinct_subjects, COUNT(*) as total_rows
FROM shopping_assistant.raw_logs.sessions_raw s
JOIN shopping_assistant.raw_logs.customer_pii c ON s.subject_ref = c.subject_ref
WHERE s.subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

UNION ALL

SELECT 'interactions', COUNT(DISTINCT i.subject_ref), COUNT(*)
FROM shopping_assistant.raw_logs.ai_interactions_raw i
JOIN shopping_assistant.raw_logs.customer_pii c ON i.subject_ref = c.subject_ref
WHERE i.subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT subject_id, subject_ref, created_at
FROM shopping_assistant.raw_logs.customer_pii
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'
ORDER BY created_at

In [0]:
%sql
SELECT session_id, started_at
FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'

In [0]:
%sql
SELECT trace_id, session_id, request_timestamp, input_text_sanitised, status
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'
ORDER BY request_timestamp

In [0]:
%sql
SELECT session_id, started_at
FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'

In [0]:
%sql
SELECT session_id, started_at FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'

In [0]:
%sql
SELECT trace_id, request_timestamp, input_text_sanitised, status
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'
ORDER BY request_timestamp

In [0]:
%sql
SELECT trace_id, run_id, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

In [0]:
%sql
SELECT trace_id, policy_name, result
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id IN (
    'e1df8cf2-8fde-47b1-82d6-43fea645a139',
    'a2f06512-7a3c-4ee0-8a3d-7794dab957d0',
    '22bb5d6c-e421-4f1d-95bc-de1ac36ad81f',
    '7b961a3e-bdf0-4be8-9636-54cd14bfd21a',
    'e5077dca-2124-44a8-8ac8-e22c210fa414',
    '7e97c037-d88a-4fe8-bd56-2380266ea3be',
    '140c12b7-37be-42fd-bada-4b7538e8e5b7',
    '054ffe1b-d08d-4fa9-a046-825c4c9c43f0',
    '1b297f81-829f-4d55-8792-0d87c676fc8b',
    'f385a121-98bb-4bbb-93c6-f438cf4b7b62'
)
ORDER BY trace_id

In [0]:
%sql
SELECT trace_id,
  (SELECT COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw i WHERE i.trace_id = t.trace_id) as interaction_count,
  (SELECT COUNT(*) FROM shopping_assistant.raw_logs.node_executions_raw n WHERE n.trace_id = t.trace_id) as node_count,
  (SELECT COUNT(*) FROM shopping_assistant.raw_logs.tool_calls_raw tc WHERE tc.trace_id = t.trace_id) as tool_count,
  (SELECT COUNT(*) FROM shopping_assistant.raw_logs.model_outputs_raw mo WHERE mo.trace_id = t.trace_id) as model_output_count,
  (SELECT COUNT(*) FROM shopping_assistant.raw_logs.guardrail_results_raw g WHERE g.trace_id = t.trace_id) as guardrail_count
FROM (
  SELECT trace_id FROM shopping_assistant.raw_logs.ai_interactions_raw
  ORDER BY created_at DESC LIMIT 10
) t

In [0]:
%sql
SELECT 'interaction' as source, trace_id, CAST(NULL AS STRING) as detail
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

UNION ALL

SELECT 'node', trace_id, node_name
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

UNION ALL

SELECT 'tool', trace_id, tool_name
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

UNION ALL

SELECT 'model_output', trace_id, CAST(NULL AS STRING)
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

UNION ALL

SELECT 'guardrail', trace_id, policy_name
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

ORDER BY source

In [0]:
%sql
SELECT trace_id FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.audits.guardrail_results

In [0]:
%sql
SELECT trace_id, policy_name, result, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = '34bf15b7-2856-4afd-8e9e-c05a31d51bbf'

In [0]:
%sql
SELECT trace_id, policy_name, result, score, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = 'b1880d2f-6fc8-4ee6-80c5-7eef54d70ac9' AND policy_name = 'output_content_safety'

In [0]:
%sql
SELECT trace_id, status FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, policy_name, result, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = '22477ac2-6f56-47ad-bf87-2cd7a6d5c8c2'

In [0]:
%sql
SELECT trace_id FROM shopping_assistant.raw_logs.node_executions_raw WHERE trace_id = '22477ac2-6f56-47ad-bf87-2cd7a6d5c8c2'

In [0]:
%sql
SELECT trace_id FROM shopping_assistant.raw_logs.tool_calls_raw WHERE trace_id = '22477ac2-6f56-47ad-bf87-2cd7a6d5c8c2'

In [0]:
%sql
SELECT trace_id FROM shopping_assistant.raw_logs.model_outputs_raw WHERE trace_id = '22477ac2-6f56-47ad-bf87-2cd7a6d5c8c2'

In [0]:
%sql
SELECT trace_id, node_name, status
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '22477ac2-6f56-47ad-bf87-2cd7a6d5c8c2'

In [0]:
%sql
SELECT trace_id, policy_name, result, score, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = 'a109ea1d-75b4-4725-aa25-115cffd23049'

In [0]:
import os
os.environ["AUDIT_SECRET_SCOPE"] = "audit_trail_secrets"
os.environ["DATABRICKS_SQL_WAREHOUSE_ID"] = "8eba55b5eb828697"
os.environ["DATABRICKS_HOST"] = "https://adb-4206962778623078.18.azuredatabricks.net"

import importlib
import audit_wrapper
importlib.reload(audit_wrapper)
from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(app_id="myre_app", catalog="shopping_assistant")

raw_text = """You're looking for black bags, and I've found three great options for you.
**Nylon Adventure Overnight Bag in Black** by PIERRE CARDIN **$148.50**
This travel bag is designed for adventure-seekers, featuring a sturdy nylon construction that can withstand rough handling. With multiple compartments, you'll have ample space to organize your gear, and the nylon material provides durability for frequent trips.
**Leather Tote Bag in Black** by PIERRE CARDIN **$219.95**
This high-quality leather tote is perfect for work or daily commuting, offering a spacious interior with a dedicated laptop sleeve. The solid leather construction exudes professionalism, while the bag's comfortable straps ensure all-day wear.
**Susy Soft Lge Hobo Bag in Black** by Armani Exchange **$217.00**
This stylish hobo bag features a soft polyester material and a classic design, making it ideal for everyday use. With ample space inside, you can carry your essentials in comfort, and the solid pattern ensures a timeless look."""

computed_hash = wrapper._hmac(raw_text)
print("Computed hash:", computed_hash)
print("Stored hash:  ", "be3eed7f18b8f539e4a5b85a50106275ebc48c45f0d7de3c84d63f7f58a28207")
print("Match:", computed_hash == "be3eed7f18b8f539e4a5b85a50106275ebc48c45f0d7de3c84d63f7f58a28207")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
%sql
SELECT trace_id, policy_name, result, score, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = 'b1880d2f-6fc8-4ee6-80c5-7eef54d70ac9' AND policy_name = 'input_content_safety'

In [0]:
result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = """
        SELECT output_text_sanitised, output_hash 
        FROM shopping_assistant.raw_logs.model_outputs_raw
        WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'
    """,
    wait_timeout = "10s",
)
stored_text = result.result.data_array[0][0]
stored_hash = result.result.data_array[0][1]

computed_hash = wrapper._hmac(stored_text)

print("Computed hash:", computed_hash)
print("Stored hash:  ", stored_hash)
print("Match:", computed_hash == stored_hash)

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = 'a109ea1d-75b4-4725-aa25-115cffd23049'

In [0]:
%sql
SELECT trace_id, status FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, output_text_sanitised, contains_pii_flag, pii_types_found
FROM shopping_assistant.raw_logs.model_outputs_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, recommended_items
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = 'af9e1abf-eff5-4aee-be2e-3f064f073454'

In [0]:
%sql
SELECT trace_id, output_text_sanitised, output_hash, output_type, finish_reason, contains_pii_flag
FROM shopping_assistant.raw_logs.model_outputs_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql SHOW SCHEMAS IN shopping_assistant

In [0]:
%sql
SELECT trace_id, tool_name, status, tool_inputs, tool_outputs, latency_ms
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = 'b1880d2f-6fc8-4ee6-80c5-7eef54d70ac9'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE trace_id = 'c9c1f497-bce0-49bd-809f-07c5e53d5772'

In [0]:
%sql
SELECT trace_id, tool_name, status, tool_outputs
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = 'c9c1f497-bce0-49bd-809f-07c5e53d5772'

In [0]:
%sql
SELECT trace_id, input_text_sanitised, status, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, tool_name, status, tool_outputs
FROM shopping_assistant.raw_logs.tool_calls_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT
    trace_id,
    node_name,
    node_type,
    status,
    latency_ms,
    node_order,
    input_summary,
    output_summary,
    node_metadata,
    started_at
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '0ccd4de5-189c-47f0-8227-f4c3efff1abc'
ORDER BY started_at;

In [0]:
%sql
SELECT
    trace_id,
    tool_name,
    status,
    error_message,
    called_at
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '0ccd4de5-189c-47f0-8227-f4c3efff1abc'
  AND tool_name = 'product_search'
ORDER BY called_at;

In [0]:
%sql
SELECT
    trace_id,
    tool_name,
    status,
    error_message,
    called_at
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '2c6be67f-280f-4886-8527-8c2005cae535'
  AND tool_name = 'product_search'
ORDER BY called_at;

In [0]:
%sql
SELECT
    trace_id,
    request_timestamp
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY request_timestamp DESC
LIMIT 10;

In [0]:
%sql
SELECT
    trace_id,
    tool_name,
    status,
    error_message,
    called_at
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '<YOUR_TRACE_ID>'
  AND tool_name = 'product_search'
ORDER BY called_at;

In [0]:
%sql
SELECT
    trace_id,
    output_id,
    output_type,
    output_hash,
    finish_reason,
    created_at
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '5e48229a-3037-4161-9621-897180b94d73';v

In [0]:
%sql
SELECT
    trace_id,
    node_name,
    status,
    error_message,
    latency_ms,
    started_at
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '5e48229a-3037-4161-9621-897180b94d73'
ORDER BY started_at;

In [0]:
%sql
SELECT
    trace_id,
    tool_name,
    status,
    error_message,
    LENGTH(error_message) AS error_length,
    called_at
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '5e48229a-3037-4161-9621-897180b94d73'
  AND tool_name = 'product_search';

In [0]:
%sql
SELECT
    trace_id,
    tool_name,
    status,
    error_message,
    called_at
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE status = 'error'
ORDER BY called_at DESC
LIMIT 20;

In [0]:
%sql
SELECT
    trace_id,
    node_name,
    node_order,
    node_type,
    status,
    latency_ms,
    subject_ref,
    started_at
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '2c6be67f-280f-4886-8527-8c2005cae535'
ORDER BY node_order;

In [0]:
%sql
SELECT
    node_name,
    node_type,
    COUNT(*) AS execution_count
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d'
GROUP BY node_name, node_type
ORDER BY node_name;

In [0]:
%sql
SELECT *
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY request_timestamp DESC
LIMIT 1;

In [0]:
%sql
SELECT 'ai_interactions_raw' AS table_name, COUNT(*) AS records
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d'

UNION ALL

SELECT 'node_executions_raw', COUNT(*)
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d'

UNION ALL

SELECT 'tool_calls_raw', COUNT(*)
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d'

UNION ALL

SELECT 'model_outputs_raw', COUNT(*)
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d'

UNION ALL

SELECT 'guardrail_results_raw', COUNT(*)
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = 'a56e7da9-1769-4b09-8ddb-fd30c6a6516d';

In [0]:
%sql
SELECT trace_id, created_at FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'

In [0]:
%sql
SELECT trace_id, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = '5bdd76d3-cd0f-4e8c-8a4b-b7265017fb88'

In [0]:
%sql
SELECT trace_id, output_type, created_at
FROM shopping_assistant.raw_logs.model_outputs_raw
ORDER BY created_at DESC LIMIT 10

In [0]:
%sql
SELECT COUNT(*) as total_rows, MAX(created_at) as most_recent
FROM shopping_assistant.raw_logs.model_outputs_raw

In [0]:
if self.audit_wrapper:
    print(f"[DEBUG] About to log model output for trace_id={_current_trace_id.get()}")
    products = state.get("reranked_results") or state.get("results") or []

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'

In [0]:
%sql
SELECT 'interaction' as source, trace_id, CAST(NULL AS STRING) as detail
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'

UNION ALL

SELECT 'node', trace_id, node_name
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'
UNION ALL
SELECT 'tool', trace_id, tool_name
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'
UNION ALL
SELECT 'model_output', trace_id, CAST(NULL AS STRING)
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'
UNION ALL
SELECT 'guardrail', trace_id, policy_name
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = '78795baa-1ab2-41a9-912b-bf82644b3864'
ORDER BY source

In [0]:
%sql
SELECT trace_id, session_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, policy_name, result, triggered_block
FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE trace_id = '55e0b214-402b-4007-988c-b7bedf0ac335'

In [0]:
%sql
SELECT trace_id, session_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, output_text_sanitised, contains_pii_flag, pii_types_found
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '00656b1a-2951-41b1-b26e-755b18b1101f'

In [0]:
%sql
SELECT trace_id, session_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE trace_id = 'cc8d3a5f-c054-44b9-b494-fbb63a998eae'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE failed_table = 'model_outputs_raw'
ORDER BY occurred_at DESC LIMIT 5

In [0]:
%sql
SELECT trace_id, session_id, input_text_sanitised, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, output_text_sanitised, contains_pii_flag, pii_types_found
FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = 'cc8d3a5f-c054-44b9-b494-fbb63a998eae'

In [0]:
%sql
SELECT trace_id, session_id, status, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT 'node_executions_raw' as tbl, trace_id, node_name, input_summary, output_summary
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id IN ('dacb661d-b177-41c9-b058-0e6bebf11c68', '6b42d4a6-0ddf-4538-95f0-26dd4f0d7f86')

In [0]:
%sql
SELECT trace_id, node_name, input_summary, output_summary
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id IN ('dacb661d-b177-41c9-b058-0e6bebf11c68', '6b42d4a6-0ddf-4538-95f0-26dd4f0d7f86')

In [0]:
%sql
SELECT 'tool_calls_raw' as tbl, trace_id, tool_name, tool_inputs, tool_outputs
FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE trace_id IN ('dacb661d-b177-41c9-b058-0e6bebf11c68', '6b42d4a6-0ddf-4538-95f0-26dd4f0d7f86')

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 15

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 10

In [0]:
%sql
SELECT DISTINCT subject_ref FROM shopping_assistant.raw_logs.sessions_raw
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT DISTINCT subject_ref FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT DISTINCT subject_ref FROM shopping_assistant.raw_logs.node_executions_raw
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT DISTINCT subject_ref FROM shopping_assistant.raw_logs.customer_pii
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'

In [0]:
%sql
SELECT 'sessions_raw' as tbl, COUNT(*) as hits FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_metadata LIKE '%@%.%'
UNION ALL
SELECT 'ai_interactions_raw', COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE input_text_sanitised LIKE '%@%.%'
UNION ALL
SELECT 'node_executions_raw', COUNT(*) FROM shopping_assistant.raw_logs.node_executions_raw
WHERE input_summary LIKE '%@%.%' OR output_summary LIKE '%@%.%'
UNION ALL
SELECT 'tool_calls_raw', COUNT(*) FROM shopping_assistant.raw_logs.tool_calls_raw
WHERE tool_inputs LIKE '%@%.%' OR tool_outputs LIKE '%@%.%'
UNION ALL
SELECT 'model_outputs_raw', COUNT(*) FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE output_text_sanitised LIKE '%@%.%'
UNION ALL
SELECT 'guardrail_results_raw', COUNT(*) FROM shopping_assistant.raw_logs.guardrail_results_raw
WHERE policy_name LIKE '%@%.%'
UNION ALL
SELECT 'logging_failures', COUNT(*) FROM shopping_assistant.raw_logs.logging_failures
WHERE error_message LIKE '%@%.%'

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 2

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 2

In [0]:
%sql
SELECT trace_id, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 2

In [0]:
%sql
SELECT trace_id, input_text_sanitised, input_hash
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, session_id, status, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT node_execution_id, trace_id, node_name, status, latency_ms, subject_ref, node_order, node_type
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE trace_id = '1b4d0181-0f2b-42e8-8556-819dfdbd7a42'
ORDER BY started_at

In [0]:
%sql
SELECT trace_id, app_id, session_id, latency_ms, request_timestamp
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE latency_ms IS NULL
ORDER BY created_at DESC

In [0]:
%sql
SELECT trace_id, confidence_score, latency_ms
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC
LIMIT 10

In [0]:
%sql
SELECT trace_id, confidence_score, latency_ms
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC
LIMIT 10

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = '91f62e88-5f3c-4048-a6eb-388713db8c61'

In [0]:
%sql
SELECT trace_id, COUNT(*) as row_count, input_text_sanitised
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE created_at >= current_timestamp() - INTERVAL 10 MINUTES
GROUP BY trace_id, input_text_sanitised
ORDER BY MAX(created_at) DESC

In [0]:
%sql
SELECT trace_id, status, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC
LIMIT 10

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = 'bc0a67e4-1c29-4263-a75d-9d843db3ceba'

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = '91f62e88-5f3c-4048-a6eb-388713db8c61'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.model_outputs_raw
WHERE trace_id = '3219d79d-39e1-4f41-b035-f1a1011e39c3'

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE trace_id = '91f62e88-5f3c-4048-a6eb-388713db8c61'

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT trace_id, session_id, status, app_metadata
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT session_id, subject_ref, started_at, channel, device_type
FROM shopping_assistant.raw_logs.sessions_raw
WHERE subject_ref = '356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb'
ORDER BY started_at DESC

In [0]:
%sql
SELECT trace_id, session_id, app_id, status, request_timestamp
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'
ORDER BY request_timestamp

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_id = 'fad071a2-74f5-4341-86c6-b9d020790156'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.sessions_raw
ORDER BY created_at DESC LIMIT 1

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.sessions_raw
WHERE session_id = 'b1b283cf-a2bc-41a9-8385-1c7f7c769345'

In [0]:
%sql
SELECT node_name, status, node_metadata, created_at
FROM shopping_assistant.raw_logs.node_executions_raw
ORDER BY created_at DESC
LIMIT 20;

In [0]:
%sql
SELECT node_name, status, node_metadata, created_at
FROM shopping_assistant.raw_logs.node_executions_raw
ORDER BY created_at DESC
LIMIT 5;

In [0]:
%sql
SELECT node_name, status, node_metadata, created_at
FROM shopping_assistant.raw_logs.node_executions_raw
ORDER BY created_at DESC
LIMIT 20;

In [0]:
%sql
SELECT 
    node_name,
    node_metadata:agent_id       AS agent_id,
    node_metadata:agent_role     AS agent_role,
    node_metadata:handoff_from   AS handoff_from,
    node_metadata:handoff_reason AS handoff_reason,
    node_metadata:reasoning      AS reasoning
FROM shopping_assistant.raw_logs.node_executions_raw
WHERE node_metadata IS NOT NULL
ORDER BY created_at DESC
LIMIT 5;

In [0]:
spark.sql("DESCRIBE shopping_assistant.raw_logs.node_executions_raw").display()

In [0]:
spark.sql("DESCRIBE shopping_assistant.raw_logs.node_executions_raw")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")
print("Restored")

In [0]:
import hmac as hmac_lib, hashlib, base64

def load(scope):
    s = w.secrets.get_secret(scope=scope, key="hmac_key_myre_app")
    try:
        return base64.b64decode(s.value).decode("utf-8")
    except Exception:
        return s.value

k1 = load("shopping_assistant")
k2 = load("audit_trail_secrets")
print("identical:", k1 == k2)

In [0]:
print([k.key for k in w.secrets.list_secrets(scope="audit_trail_secrets")])

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

print([s.name for s in w.secrets.list_scopes()])
print([k.key for k in w.secrets.list_secrets(scope="shopping_assistant")])
print(w.secrets.list_acls(scope="shopping_assistant"))

In [0]:
%sql
SELECT
  trace_id,
  request_timestamp,
  input_text_sanitised,
  status,
  latency_ms,
  subject_ref
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY request_timestamp DESC
LIMIT 20;

In [0]:
%sql
-- Latest interaction + what each table logged for it
WITH latest AS (
  SELECT trace_id, request_timestamp, status, latency_ms, subject_ref
  FROM shopping_assistant.raw_logs.ai_interactions_raw
  ORDER BY request_timestamp DESC
  LIMIT 1
)
SELECT
  l.trace_id,
  l.request_timestamp,
  l.status,
  l.latency_ms,
  (SELECT count(*) FROM shopping_assistant.raw_logs.node_executions_raw   n WHERE n.trace_id = l.trace_id) AS node_rows,
  (SELECT count(*) FROM shopping_assistant.raw_logs.guardrail_results_raw g WHERE g.trace_id = l.trace_id) AS guardrail_rows,
  (SELECT count(*) FROM shopping_assistant.raw_logs.tool_calls_raw        t WHERE t.trace_id = l.trace_id) AS tool_rows,
  (SELECT count(*) FROM shopping_assistant.raw_logs.model_outputs_raw     m WHERE m.trace_id = l.trace_id) AS output_rows
FROM latest l;

In [0]:
w.secrets.delete_secret(scope="shopping_assistant", key="audit-app-id")
print("Deleted — AUDIT_APP_ID now unresolvable")

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.sessions_raw
ORDER BY created_at DESC
LIMIT 3

In [0]:
result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = f"""
        SELECT email, user_country, regulation FROM shopping_assistant.raw_logs.customer_pii
        WHERE email = '{new_email}'
    """,
    wait_timeout = "10s",
)
print("State:", result.status.state)
print("Error:", result.status.error)
if result.result:
    print("Data:", result.result.data_array)
else:
    print("No result object — query likely still running or failed")

In [0]:
result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = f"""
        SELECT email, user_country, regulation FROM shopping_assistant.raw_logs.customer_pii
        WHERE email = '{new_email}'
    """,
    wait_timeout = "10s",
)
print(result.result.data_array)

In [0]:
import os
os.environ["AUDIT_SECRET_SCOPE"] = "audit_trail_secrets"
import importlib
import audit_wrapper
importlib.reload(audit_wrapper)
from audit_wrapper import AuditWrapper
wrapper3 = AuditWrapper(app_id="myre_app", catalog="shopping_assistant")
computed_id, computed_ref = wrapper3._compute_refs(new_email)
print("Computed subject_id: ", computed_id)
print("Computed subject_ref:", computed_ref)
print("Match subject_id?  ", computed_id == "111f7dd4fdd68cfceef00b2313d3652043bb7ac8d5f849e17d0f7d1e0251ee84")
print("Match subject_ref? ", computed_ref == "356b376217175cec553856d4772a9c8e89d02d668cd2f40e72f84e5075ad54fb")

In [0]:
new_email = "p9475708@gmail.com"

result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = f"""
        SELECT subject_id, subject_ref FROM shopping_assistant.raw_logs.customer_pii
        WHERE email = '{new_email}'
    """,
    wait_timeout = "10s",
)
if result.result and result.result.data_array:
    stored_id  = result.result.data_array[0][0]
    stored_ref = result.result.data_array[0][1]
    print("Stored subject_id: ", stored_id)
    print("Stored subject_ref:", stored_ref)
else:
    print("No row found yet — wait a few seconds and retry")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit_secret_scope", string_value="audit_trail_secrets")
print("Fixed — AUDIT_SECRET_SCOPE now correctly set to audit_trail_secrets")

In [0]:
new_email = "PASTE_THE_NEW_EMAIL_HERE"

result = w.statement_execution.execute_statement(
    warehouse_id = "8eba55b5eb828697",
    statement = f"""
        SELECT subject_id, subject_ref FROM shopping_assistant.raw_logs.customer_pii
        WHERE email = '{new_email}'
    """,
    wait_timeout = "10s",
)
if result.result and result.result.data_array:
    stored_id  = result.result.data_array[0][0]
    stored_ref = result.result.data_array[0][1]
    print("Stored subject_id: ", stored_id)
    print("Stored subject_ref:", stored_ref)
else:
    print("No row found yet — wait a few seconds and retry")scope_secret = w.secrets.get_secret(scope="shopping_assistant", key="audit_secret_scope")
print(scope_secret.value if not scope_secret.value.startswith("eyJ") else "base64 - decode needed")

In [0]:
import os
print(repr(os.getenv("AUDIT_SECRET_SCOPE")))

In [0]:
importlib.reload(audit_wrapper)
from audit_wrapper import AuditWrapper
wrapper2 = AuditWrapper(app_id="myre_app", catalog="shopping_assistant")
subject_id, subject_ref = wrapper2._compute_refs("user2test777789@gmail.com")
print(subject_id == "9804cfe37062d081f352e9829780be8d10357e40b8f2f02d9a872a32df4d4648")

In [0]:
scope_secret = w.secrets.get_secret(scope="shopping_assistant", key="audit_secret_scope")
try:
    print(base64.b64decode(scope_secret.value).decode("utf-8"))
except:
    print(scope_secret.value)

In [0]:
key_secret2 = w.secrets.get_secret(scope="shopping_assistant", key="hmac_key_myre_app")
try:
    raw_key2 = base64.b64decode(key_secret2.value).decode("utf-8")
except Exception:
    raw_key2 = key_secret2.value
key_bytes2 = raw_key2.encode()

test_subject_id2 = hmac_lib.new(key_bytes2, email.lower().strip().encode(), hashlib.sha256).hexdigest()
test_subject_ref2 = hmac_lib.new(key_bytes2, test_subject_id2.encode(), hashlib.sha256).hexdigest()

print("Using shopping_assistant scope key:")
print("subject_id: ", test_subject_id2)
print("subject_ref:", test_subject_ref2)
print()
print("Match stored subject_id? ", test_subject_id2 == "9804cfe37062d081f352e9829780be8d10357e40b8f2f02d9a872a32df4d4648")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
import hmac as hmac_lib
import hashlib
import base64

key_secret = w.secrets.get_secret(scope="audit_trail_secrets", key="hmac_key_myre_app")
try:
    raw_key = base64.b64decode(key_secret.value).decode("utf-8")
except Exception:
    raw_key = key_secret.value
key_bytes = raw_key.encode()

email = "user2test777789@gmail.com"
computed_subject_id = hmac_lib.new(key_bytes, email.lower().strip().encode(), hashlib.sha256).hexdigest()
computed_subject_ref = hmac_lib.new(key_bytes, computed_subject_id.encode(), hashlib.sha256).hexdigest()

print("Computed subject_id:", computed_subject_id)
print("Computed subject_ref:", computed_subject_ref)

In [0]:
print("raw_key type:", type(raw_key))
print("raw_key length:", len(raw_key))
print("raw_key repr:", repr(raw_key)[:50])

In [0]:
%sql
SELECT subject_id, subject_ref FROM shopping_assistant.raw_logs.customer_pii
WHERE email = 'user2test777789@gmail.com'

In [0]:
import sys
sys.path.append("/Workspace/Repos/nupur@xponent.ai/shopping-assistant/python/source_code")

import os
os.environ["DATABRICKS_SQL_WAREHOUSE_ID"] = "8eba55b5eb828697"
os.environ["DATABRICKS_HOST"] = "https://adb-4206962778623078.18.azuredatabricks.net"

import importlib
import audit_wrapper
importlib.reload(audit_wrapper)
from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(app_id="myre_app", catalog="shopping_assistant")

subject_id, subject_ref = wrapper._compute_refs("user2test777789@gmail.com")
print("Wrapper subject_id: ", subject_id)
print("Wrapper subject_ref:", subject_ref)

print()
print("Match subject_id?  ", subject_id == "9804cfe37062d081f352e9829780be8d10357e40b8f2f02d9a872a32df4d4648")
print("Match subject_ref? ", subject_ref == "091223f312fb8af2efcc3ca28729dc8bcf8e650e6be0b9288a5b796f29001008")

In [0]:
for k in w.secrets.list_secrets(scope="audit_trail_secrets"):
    if k.key == "hmac_key_myre_app":
        print("Key:", k.key)
        print("Last updated (epoch ms):", k.last_updated_timestamp)
        import datetime
        print("Last updated:", datetime.datetime.fromtimestamp(k.last_updated_timestamp/1000))

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE failed_table = 'customer_pii'
ORDER BY occurred_at DESC
LIMIT 5

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.customer_pii
WHERE email = 'user2test777789@gmail.com'

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.customer_pii
ORDER BY created_at DESC
LIMIT 5

In [0]:
import os
os.environ["AUDIT_SECRET_SCOPE"] = "nonexistent_scope_xyz"  # doesn't exist
os.environ["DATABRICKS_SQL_WAREHOUSE_ID"] = "8eba55b5eb828697"
os.environ["DATABRICKS_HOST"] = "https://adb-4206962778623078.18.azuredatabricks.net"

import importlib
import audit_wrapper
importlib.reload(audit_wrapper)  # reload to pick up fresh env vars
from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(app_id="myre_app", catalog="shopping_assistant")

result = wrapper.log_interaction(
    user_email="test_fallback_scope@example.com",
    user_input="test query for AT-INIT-07",
    model_output="test response",
    model_name="test-model",
    status="success",
    trace_id="test-trace-fallback-001",
    session_id="test-session-fallback-001",
    user_country="IN",
)
print(result)

In [0]:
import os
os.environ["DATABRICKS_SQL_WAREHOUSE_ID"] = "8eba55b5eb828697"
os.environ["DATABRICKS_HOST"] = "https://adb-4206962778623078.18.azuredatabricks.net"

import sys
sys.path.append("/Workspace/Repos/nupur@xponent.ai/shopping-assistant/python/source_code")

from audit_wrapper import AuditWrapper

wrapper = AuditWrapper(app_id="shopping_assistant_app", catalog="shopping_assistant")

In [0]:
wrapper.log_interaction(
    user_email="test_hmac_missing@example.com",
    user_input="test query for AT-INIT-06",
    model_output="test response",
    model_name="test-model",
    status="success",
    trace_id="test-trace-001",
    session_id="test-session-001",
    user_country="IN",
)

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
WHERE app_id = 'shopping_assistant_app'
ORDER BY occurred_at DESC LIMIT 5

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
w.secrets.delete_secret(scope="audit_trail_secrets", key="hmac_key_shopping_assistant_app")
print("Deleted — HMAC key missing for shopping_assistant_app")

In [0]:
for k in w.secrets.list_secrets(scope="audit_trail_secrets"):
    print(k.key)

In [0]:
%sql
SELECT COUNT(*) as row_count
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE app_id = 'shopping_assistant_app'

In [0]:
%sql
SELECT COUNT(*) as row_count
FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE app_id = 'shopping_assistant_app'

In [0]:
%sql
SELECT COUNT(*) as row_count
FROM shopping_assistant.raw_logs.customer_pii
WHERE regulation IS NOT NULL

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.customer_pii
WHERE subject_id IN (
    SELECT DISTINCT subject_ref FROM shopping_assistant.raw_logs.ai_interactions_raw 
    WHERE app_id = 'shopping_assistant_app'
);

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="databricks_sql_warehouse_id", string_value="8eba55b5eb828697")

In [0]:
%sql
SELECT trace_id, session_id, app_id, status, created_at
FROM shopping_assistant.raw_logs.ai_interactions_raw
ORDER BY created_at DESC
LIMIT 5

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="databricks_sql_warehouse_id", string_value="8eba55b5eb828697")

In [0]:
row = Row(
    registry_entry_id = str(uuid.uuid4()),
    app_id            = "myre_app",
    app_name          = "Myre Shopping Assistant",
    app_owner_email   = "venkat@xponent.ai",
    business_domain   = "retail",
    regulations       = '["GDPR", "DPDP"]',
    secret_key_name   = "hmac_key_myre_app",
    status            = "active",
    onboarded_at      = datetime.now(timezone.utc),
    entry_created_at  = datetime.now(timezone.utc),
    app_metadata      = None,
    schema_version    = "1.0",
)

spark.createDataFrame([row], schema=schema).write.format("delta").mode("append") \
    .saveAsTable("shopping_assistant.raw_logs.app_registry")

print("Reactivated")

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.logging_failures
ORDER BY occurred_at DESC
LIMIT 5

In [0]:
display(spark.sql("""
    SELECT trace_id, session_id, app_id, status, created_at
    FROM shopping_assistant.raw_logs.ai_interactions_raw
    ORDER BY created_at DESC
    LIMIT 5
"""))

In [0]:
row = Row(
    registry_entry_id = str(uuid.uuid4()),
    app_id            = "myre_app",
    app_name          = "Myre Shopping Assistant",
    app_owner_email   = "venkat@xponent.ai",
    business_domain   = "retail",
    regulations       = '["GDPR", "DPDP"]',
    secret_key_name   = "hmac_key_myre_app",
    status            = "active",
    onboarded_at      = datetime.now(timezone.utc),
    entry_created_at  = datetime.now(timezone.utc),
    app_metadata      = None,
    schema_version    = "1.0",
)

spark.createDataFrame([row], schema=schema).write.format("delta").mode("append") \
    .saveAsTable("shopping_assistant.raw_logs.app_registry")

print("Reactivated")

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.app_registry
WHERE app_id = 'myre_app'
ORDER BY entry_created_at DESC
LIMIT 1;

In [0]:
from datetime import datetime, timezone
import uuid
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

now = datetime.now(timezone.utc)

schema = StructType([
    StructField("registry_entry_id", StringType(), True),
    StructField("app_id", StringType(), True),
    StructField("app_name", StringType(), True),
    StructField("app_owner_email", StringType(), True),
    StructField("business_domain", StringType(), True),
    StructField("regulations", StringType(), True),
    StructField("secret_key_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("onboarded_at", TimestampType(), True),
    StructField("entry_created_at", TimestampType(), True),
    StructField("app_metadata", StringType(), True),
    StructField("schema_version", StringType(), True),
])

row = Row(
    registry_entry_id = str(uuid.uuid4()),
    app_id            = "myre_app",
    app_name          = "Myre Shopping Assistant",
    app_owner_email   = "venkat@xponent.ai",
    business_domain   = "retail",
    regulations       = '["GDPR", "DPDP"]',
    secret_key_name   = "hmac_key_myre_app",
    status            = "suspended",
    onboarded_at      = now,
    entry_created_at  = now,
    app_metadata      = None,
    schema_version    = "1.0",
)

spark.createDataFrame([row], schema=schema).write.format("delta").mode("append") \
    .saveAsTable("shopping_assistant.raw_logs.app_registry")

print("Suspended row appended")

In [0]:
display(spark.sql("""
    SELECT app_id, status, entry_created_at
    FROM shopping_assistant.raw_logs.app_registry
    WHERE app_id = 'myre_app'
    ORDER BY entry_created_at DESC
    LIMIT 1
"""))

In [0]:
spark.sql("DESCRIBE shopping_assistant.raw_logs.app_registry")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="fake_app_test_123")
print("Changed to unregistered app_id")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.ai_interactions_raw
   ORDER BY created_at DESC LIMIT 3;

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.ai_interactions_raw
   ORDER BY created_at DESC LIMIT 3;

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")

In [0]:
for k in w.secrets.list_secrets(scope="shopping_assistant"):
    print(k.key)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")
print("Secret restored")

In [0]:
for k in w.secrets.list_secrets(scope="shopping_assistant"):
    print(k.key)

In [0]:
for key in ["audit-app-id", "audit-catalog", "audit-secret-scope", "databricks_sql_warehouse_id"]:
    try:
        # can't read secret values directly via SDK for security, but confirm they exist
        pass
    except:
        pass

for k in w.secrets.list_secrets(scope="shopping_assistant"):
    print(k.key)

In [0]:
%sql
SELECT COUNT(*) FROM shopping_assistant.raw_logs.ai_interactions_raw
WHERE regulation_at_time = 'AUS'

In [0]:
import sys
sys.path.insert(0, "/dbfs/audit_trail/")
from audit_wrapper import determine_regulation
print(determine_regulation(country="AU"))   # should print AUS

In [0]:
%sql
SELECT status FROM shopping_assistant.raw_logs.app_registry
WHERE app_id = 'myre_app'
ORDER BY entry_created_at DESC
LIMIT 1;

In [0]:
%sql
SELECT app_id, app_name, status, entry_created_at
FROM shopping_assistant.raw_logs.app_registry
ORDER BY entry_created_at DESC;

In [0]:
%sql
SELECT * FROM shopping_assistant.raw_logs.app_registry
WHERE app_id = 'myre_app'
ORDER BY entry_created_at DESC
LIMIT 1;

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# find the exact scope name
for s in w.secrets.list_scopes():
    print(s.name)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

for scope_name in ["shopping_assistant", "audit_trail_secrets"]:
    print(f"--- {scope_name} ---")
    try:
        for k in w.secrets.list_secrets(scope=scope_name):
            print(" ", k.key)
    except Exception as e:
        print("  Error:", e)

In [0]:
w.secrets.delete_secret(scope="shopping_assistant", key="audit-app-id")
print("Deleted — AUDIT_APP_ID now unresolvable")

In [0]:
w.secrets.delete_secret(scope="shopping_assistant", key="audit-app-id")
print("Deleted — AUDIT_APP_ID now unresolvable")

In [0]:
for k in w.secrets.list_secrets(scope="shopping_assistant"):
    print(k.key)

In [0]:
for k in w.secrets.list_secrets(scope="shopping_assistant"):
    print(k.key)

In [0]:
w.secrets.put_secret(scope="shopping_assistant", key="audit-app-id", string_value="myre_app")